In [22]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [23]:
df = pd.read_csv('data.csv')

In [24]:
df.head()

,review,sentiment
0,Every great gangster movie has under-currents ...,positive
1,"I just saw this film last night, and I have to...",positive
2,This film is mildly entertaining if one neglec...,negative
3,Quentin Tarantino's partner in crime Roger Ava...,negative
4,I sat through this on TV hoping because of the...,negative


In [25]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


In [26]:
df = normalize_text(df)
df.head()

,review,sentiment
0,every great gangster movie under current human...,positive
1,saw film last night say loved every minute tak...,positive
2,film mildly entertaining one neglect acknowled...,negative
3,quentin tarantino s partner crime roger avary ...,negative
4,sat tv hoping name would worth time but dear g...,negative


In [27]:
df['sentiment'].value_counts()

sentiment
negative    269
positive    231
Name: count, dtype: int64

In [28]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [29]:
df.head()

,review,sentiment
0,every great gangster movie under current human...,1
1,saw film last night say loved every minute tak...,1
2,film mildly entertaining one neglect acknowled...,0
3,quentin tarantino s partner crime roger avary ...,0
4,sat tv hoping name would worth time but dear g...,0


In [30]:
df['sentiment'].value_counts()

sentiment
0    269
1    231
Name: count, dtype: int64

In [31]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [32]:
vectorizer = CountVectorizer(max_features=50)
x = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [33]:
X_train,X_text,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [35]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/ashokreddy304/Mlops-Project.mlflow')
dagshub.init(repo_owner='ashokreddy304',repo_name='Mlops-Project',mlflow=True)

mlflow.set_experiment("Logistic Regression")

2026-04-23 14:06:52,507 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/ashokreddy304/Mlops-Project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "ashokreddy304/Mlops-Project"

2026-04-23 14:06:52,520 - INFO - Initialized MLflow to track repo "ashokreddy304/Mlops-Project"


Repository ashokreddy304/Mlops-Project initialized!

2026-04-23 14:06:52,524 - INFO - Repository ashokreddy304/Mlops-Project initialized!
2026/04/23 14:06:52 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/aa7c0cb4a6de40d4ae434726735cceee', creation_time=1776933414502, experiment_id='2', last_update_time=1776933414502, lifecycle_stage='active', name='Logistic Regression', tags={}, trace_location=None, workspace='default'>

In [36]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [37]:

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_text)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")


        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-04-23 14:06:56,034 - INFO - Starting MLflow run...
2026-04-23 14:06:56,575 - INFO - Logging preprocessing parameters...
2026-04-23 14:06:57,626 - INFO - Initializing Logistic Regression model...
2026-04-23 14:06:57,630 - INFO - Fitting the model...
2026-04-23 14:06:57,686 - INFO - Model training complete.
2026-04-23 14:06:57,688 - INFO - Logging model parameters...
2026-04-23 14:06:58,000 - INFO - Making predictions...
2026-04-23 14:06:58,003 - INFO - Calculating evaluation metrics...
2026-04-23 14:06:58,045 - INFO - Logging evaluation metrics...
2026-04-23 14:06:59,339 - INFO - Saving and logging the model...
2026/04/23 14:06:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/23 14:07:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run fun-pug-866 at: https://dagshub.com/ashokreddy304/Mlops-Project.mlflow/#/experiments/2/runs/3ffa8160b6d44cd5874c7087f4c9788d
🧪 View experiment at: https://dagshub.com/ashokreddy304/Mlops-Project.mlflow/#/experiments/2
